In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cvxpy as cp
import qnt.data as qndata


# ============================================================
# 1) Load Quantiacs data + choose random liquid assets
# ============================================================
def load_quantiacs_universe(seed=7, n_assets=50):
    data = qndata.stocks.load_data()
    close = data.sel(field="close")
    is_liquid = data.sel(field="is_liquid")

    liquid_last = is_liquid.isel(time=-1).fillna(0).astype(bool)
    liquid_assets = close.asset.values[liquid_last.values]

    if len(liquid_assets) < n_assets:
        raise ValueError(f"Not enough liquid assets ({len(liquid_assets)}) to pick {n_assets}.")

    rng = np.random.default_rng(seed)
    chosen_assets = rng.choice(liquid_assets, size=n_assets, replace=False)
    return data, chosen_assets


# ============================================================
# 2) Returns, rolling mu and Sigma
# ============================================================
def compute_returns_from_close(close, use_log_returns=False):
    if use_log_returns:
        rets = np.log(close / close.shift(time=1))
    else:
        rets = close / close.shift(time=1) - 1

    rets = rets.dropna("time")
    times = rets.time.values
    R = rets.values.astype(float)
    return times, R


def compute_mu_sigma_series(data, chosen_assets, lookback=252, annualise=252, use_log_returns=False):
    close = data.sel(field="close").sel(asset=chosen_assets)
    times, R = compute_returns_from_close(close, use_log_returns=use_log_returns)

    T, n = R.shape
    mu_series = np.zeros((T, n))
    Sigma_series = np.zeros((T, n, n))

    for t in range(T):
        start = max(0, t - lookback + 1)
        window = R[start:t+1, :]

        mu = np.nanmean(window, axis=0)

        if window.shape[0] >= 2:
            Sigma = np.cov(window, rowvar=False)
            if np.ndim(Sigma) == 0:
                Sigma = np.array([[Sigma]])
        else:
            Sigma = np.eye(n) * 1e-6

        mu_series[t] = mu * annualise
        Sigma_series[t] = Sigma * annualise

    return times, mu_series, Sigma_series, R


# ============================================================
# 3) Helpers
# ============================================================
def make_psd_matrix(Sigma, ridge=1e-3, eps=1e-8):
    S = 0.5 * (Sigma + Sigma.T)
    S = S + ridge * np.eye(S.shape[0]) + eps * np.eye(S.shape[0])
    return S


def add_constraints(w, long_only=True, w_min=None, w_max=None, fully_invested=True):
    cons = []

    if fully_invested:
        cons.append(cp.sum(w) == 1)

    if long_only:
        cons.append(w >= 0)

    if w_min is not None:
        cons.append(w >= w_min)

    if w_max is not None:
        cons.append(w <= w_max)

    return cons


def solve_problem(prob):
    for solver in [cp.OSQP, cp.SCS, cp.ECOS]:
        try:
            prob.solve(solver=solver, warm_start=True, verbose=False)
            if prob.status in ["optimal", "optimal_inaccurate"]:
                return True
        except Exception:
            pass
    return False


def normalise_weights(w, fallback=None):
    if w is None:
        if fallback is None:
            raise ValueError("No solution and no fallback provided.")
        w = fallback.copy()

    w = np.asarray(w, dtype=float)
    w = np.nan_to_num(w, nan=0.0, posinf=0.0, neginf=0.0)
    w[w < 0] = 0.0

    s = w.sum()
    if s <= 0:
        if fallback is None:
            w = np.ones_like(w) / len(w)
        else:
            w = np.maximum(fallback, 0.0)
            w = w / w.sum()
    else:
        w = w / s

    return w


def turnover(W):
    return np.sum(np.abs(np.diff(W, axis=0)), axis=1)


def equity_curve(rp, start=1.0):
    return start * np.cumprod(1.0 + rp)


def max_drawdown(eq):
    peak = np.maximum.accumulate(eq)
    dd = eq / peak - 1.0
    return np.min(dd)


def summary_stats(rp, periods_per_year=252):
    rp = np.asarray(rp, dtype=float)
    rp = rp[np.isfinite(rp)]

    mean_daily = np.mean(rp)
    vol_daily = np.std(rp, ddof=1)
    ann_return = mean_daily * periods_per_year
    ann_vol = vol_daily * np.sqrt(periods_per_year)
    sharpe = ann_return / ann_vol if ann_vol > 0 else np.nan
    eq = equity_curve(rp)
    mdd = max_drawdown(eq)
    cum_return = eq[-1] - 1.0

    return {
        "n_obs": len(rp),
        "mean_daily": mean_daily,
        "vol_daily": vol_daily,
        "ann_return": ann_return,
        "ann_vol": ann_vol,
        "sharpe": sharpe,
        "max_drawdown": mdd,
        "cum_return": cum_return,
    }


def print_stats(name, stats):
    print(f"\n{name}")
    print("-" * len(name))
    for k, v in stats.items():
        if isinstance(v, (float, np.floating)):
            print(f"{k:15s}: {v: .6f}")
        else:
            print(f"{k:15s}: {v}")


# ============================================================
# 4) Time-varying transaction cost schedule
# ============================================================
def build_time_varying_cost_series(
    R,
    base_cost=0.0020,
    mode="vol_scaled",
    vol_lookback=21,
    alpha=2.0
):
    """
    Returns cost_series of shape (T,)

    Interpretation:
    - cost_series[t] is the cost coefficient used when moving into w_t

    base_cost examples:
    0.0010 = 10 bps
    0.0020 = 20 bps
    """

    T, n = R.shape

    if mode == "constant":
        return np.full(T, base_cost)

    elif mode == "vol_scaled":
        # simple cross-sectional average absolute return as market stress proxy
        market_abs = np.mean(np.abs(R), axis=1)

        vol_state = np.zeros(T)
        for t in range(T):
            start = max(0, t - vol_lookback + 1)
            vol_state[t] = np.mean(market_abs[start:t+1])

        # normalise around 1
        denom = np.nanmean(vol_state)
        if denom <= 0:
            denom = 1.0
        vol_state_norm = vol_state / denom

        cost_series = base_cost * (1.0 + alpha * (vol_state_norm - 1.0))
        cost_series = np.maximum(cost_series, 0.0001)
        return cost_series

    else:
        raise ValueError("mode must be 'constant' or 'vol_scaled'")


# ============================================================
# 5) One-step optimisation with varying lambda_t
# ============================================================
def solve_one_step_varying_cost(
    mu,
    Sigma,
    w_prev,
    gamma=5.0,
    lam_t=0.01,
    w_max=0.10,
    ridge=1e-3
):
    n = len(mu)
    w = cp.Variable(n)

    S = make_psd_matrix(Sigma, ridge=ridge)
    tc_penalty = cp.norm1(w - w_prev)

    obj = cp.Maximize(mu @ w - gamma * cp.quad_form(w, S) - lam_t * tc_penalty)
    prob = cp.Problem(obj, add_constraints(w, long_only=True, w_max=w_max, fully_invested=True))

    ok = solve_problem(prob)
    if not ok:
        return normalise_weights(w_prev)

    return normalise_weights(w.value, fallback=w_prev)


def run_one_step_varying_cost(
    mu_series,
    Sigma_series,
    cost_series,
    gamma=5.0,
    w_max=0.10,
    ridge=1e-3,
    w0=None
):
    T, n = mu_series.shape

    if w0 is None:
        w_prev = np.ones(n) / n
    else:
        w_prev = normalise_weights(w0)

    W = np.zeros((T, n))

    for t in range(T):
        w_t = solve_one_step_varying_cost(
            mu=mu_series[t],
            Sigma=Sigma_series[t],
            w_prev=w_prev,
            gamma=gamma,
            lam_t=cost_series[t],
            w_max=w_max,
            ridge=ridge
        )
        W[t] = w_t
        w_prev = w_t

    return W


# ============================================================
# 6) Gross and net realised portfolio returns
# ============================================================
def realised_returns_gross(R, W):
    """
    Uses W[t-1] on R[t]
    """
    return np.sum(R[1:] * W[:-1], axis=1)


def realised_returns_net(R, W, cost_series):
    """
    Net returns after subtracting cost of rebalancing.

    At date t:
    - gross return uses W[t-1] on R[t]
    - trading cost for entering W[t-1] is charged using turnover from W[t-2] -> W[t-1]

    So net series has length T-1, aligned with gross returns.
    """
    gross = realised_returns_gross(R, W)

    T = W.shape[0]
    trade_costs = np.zeros(T - 1)

    # for gross return at index k, corresponding market move is R[k+1]
    # weights used are W[k]
    # cost charged is for move into W[k], i.e. from W[k-1] to W[k]
    for k in range(1, T - 1):
        trn = np.sum(np.abs(W[k] - W[k - 1]))
        trade_costs[k] = cost_series[k] * trn

    net = gross - trade_costs
    return gross, net, trade_costs


# ============================================================
# 7) Main
# ============================================================
if __name__ == "__main__":
    seed = 7
    n_assets = 50
    lookback = 252

    gamma = 5.0
    w_max = 0.10
    ridge = 1e-3
    use_log_returns = False

    # transaction cost settings
    base_cost = 0.0020      # 20 bps
    cost_mode = "vol_scaled"   # "constant" or "vol_scaled"
    vol_lookback = 21
    alpha = 2.0

    # load data
    data, chosen_assets = load_quantiacs_universe(seed=seed, n_assets=n_assets)
    print("Chosen assets (first 10):", chosen_assets[:10], "... total:", len(chosen_assets))

    # returns and moments
    times, mu_series, Sigma_series, R_daily = compute_mu_sigma_series(
        data=data,
        chosen_assets=chosen_assets,
        lookback=lookback,
        annualise=252,
        use_log_returns=use_log_returns
    )

    # varying cost series
    cost_series = build_time_varying_cost_series(
        R_daily,
        base_cost=base_cost,
        mode=cost_mode,
        vol_lookback=vol_lookback,
        alpha=alpha
    )

    # run one-step strategy
    W = run_one_step_varying_cost(
        mu_series=mu_series,
        Sigma_series=Sigma_series,
        cost_series=cost_series,
        gamma=gamma,
        w_max=w_max,
        ridge=ridge
    )

    # turnover
    to = turnover(W)
    print("\nAverage turnover")
    print("----------------")
    print(np.mean(to))

    # gross and net returns
    rp_gross, rp_net, tc_paid = realised_returns_net(R_daily, W, cost_series)

    eq_gross = equity_curve(rp_gross)
    eq_net = equity_curve(rp_net)

    # stats
    stats_gross = summary_stats(rp_gross)
    stats_net = summary_stats(rp_net)

    print_stats("Gross stats", stats_gross)
    print_stats("Net stats", stats_net)

    print("\nAverage cost coefficient")
    print("------------------------")
    print(np.mean(cost_series))

    print("\nAverage realised trading cost per period")
    print("----------------------------------------")
    print(np.mean(tc_paid))

    # ========================================================
    # plots
    # ========================================================
    plt.figure(figsize=(10, 4))
    plt.plot(cost_series)
    plt.title("Time-varying cost coefficient")
    plt.xlabel("t")
    plt.ylabel("cost coefficient")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 4))
    plt.plot(to)
    plt.title("Turnover per rebalance")
    plt.xlabel("t")
    plt.ylabel("sum |Δw|")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 4))
    plt.imshow(W.T, aspect="auto")
    plt.title("One-step weights heatmap")
    plt.xlabel("t")
    plt.ylabel("asset")
    plt.colorbar(label="weight")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 4))
    plt.plot(tc_paid)
    plt.title("Realised trading cost paid each period")
    plt.xlabel("t")
    plt.ylabel("cost")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 4))
    plt.plot(eq_gross, label="Gross equity")
    plt.plot(eq_net, label="Net equity")
    plt.title("Gross vs net equity")
    plt.xlabel("t")
    plt.ylabel("equity")
    plt.legend()
    plt.tight_layout()
    plt.show()